# C2K query demo

There is no web UI in this project — a notebook is the interface for
exploring a stored run's knowledge graph. This notebook loads it
(`c2k/query.py`) and runs a few natural-language-ish queries against it, the
same way `c2k query "..." --run-id RUN` does from the CLI.

Run `c2k stub-run` / `c2k extract` first so a run actually has triples in
`data/db/c2k.sqlite` — this notebook doesn't create one.

In [ ]:
from pathlib import Path
import sys, os

# config/default.yaml's paths (data/db/..., data/gold/...) are relative, and
# the CLI always runs from the repo root — a notebook's cwd is wherever
# Jupyter was launched from, so fix it here before anything else touches them.
if not (Path.cwd() / "pyproject.toml").exists() and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from c2k.cli import _canonical_dir
from c2k.config import load_config
from c2k.query import build_graph, parse_query, query_graph
from c2k.store import Store

RUN_ID = "demo"  # change to whatever run you want to explore

config = load_config()
store = Store(config.paths.db, canonical_dir=_canonical_dir(config))
print(f"cwd: {Path.cwd()}")
print(f"db: {config.paths.db}")

## Load the graph

Every edge carries `(evidence_text, doc_id, char_start, char_end)` — the
provenance for a fact is never more than one hop away.

In [ ]:
graph = build_graph(store, RUN_ID)
print(f"{graph.number_of_nodes()} entities, {graph.number_of_edges()} triples")

for u, v, data in list(graph.edges(data=True))[:10]:
    print(f"{u!r:25} --[{data['relation']}]--> {v!r}")

## Run a few queries

`parse_query` is a small heuristic (find a relation name as a whole word,
treat capitalized tokens elsewhere as an entity filter) — no LLM call is
involved in query parsing, only in extraction/resolution upstream.
`query_graph` re-verifies each fact's evidence span against the canonical
text before returning it, dropping (and logging) anything stale.

In [ ]:
def show(text: str) -> None:
    relation, entity = parse_query(text)
    print(f"query: {text!r}  ->  relation={relation}, entity_guess={entity!r}")
    results = query_graph(store, RUN_ID, text)
    if not results:
        print("  (no matching facts)")
    for r in results:
        print(f"  {r.subject} --[{r.relation}]--> {r.object}")
        print(f"      {r.doc_id}[{r.char_start}:{r.char_end}]: {r.evidence_text!r}")
    print()


show("who acquired Mentor Graphics")
show("which company is headquartered in Wilsonville")
show("who founded SpaceX")

## Try your own query

Edit the cell below. If nothing comes back, check that the relation you're
asking about is one of the 14 in `config/relations.yaml` — the schema is
closed by design (no open information extraction).

In [ ]:
show("which companies partnered with OpenAI")

In [ ]:
store.close()